In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("../03_admixture_EPR_EPN/admixture_04_combine_output_assigned_clusters.tsv", sep="\t", header=0)
print(df.shape)
df.head()

(2268, 136)


,sample,id,SPECIES_ID,SITE_ID,BATCH_ID,POP,STRATA,lat,lon,K1.Q0,...,K15.Q12,K15.Q13,K15.Q14,BestK4,BestK4_Q,BestK5,BestK5_Q,BestK6,BestK6_Q,call_rate
0,E353-B1_3_6_333_001,E353-B1_3_6_333_001,EPN,ML,DSpr18-3269,333,G,49.616667,-77.750000,1.0,...,0.000010,0.001216,0.184532,K4.Central,0.912780,K5.Central,0.933588,K6.Central,0.924283,0.939222
1,E353-B1_3_6_6914_073,E353-B1_3_6_6914_073,EPN,ML,DSpr18-3269,6914,F,48.633333,-85.333333,1.0,...,0.042635,0.000010,0.094490,K4.Central,0.976894,K5.Central,0.922906,K6.Central,0.922046,0.914169
2,E353-B1_3_16_6917_081,E353-B1_3_16_6917_081,EPN,ML,DSpr18-3269,6917,E,49.000000,-90.450000,1.0,...,0.025962,0.000010,0.207579,K4.Central,0.994441,K5.Central,0.961041,K6.Central,0.954499,0.910718
3,E353-B3_3_7_6802_478,E353-B3_3_7_6802_478,EPN,CH,DSpr18-3269,6802,I,48.216667,-58.916667,1.0,...,0.002647,0.223485,0.013482,K4.East,0.778542,K5.East,0.754353,K6.East,0.749056,0.915000
4,E353-B3_3_14_6986_371,E353-B3_3_14_6986_371,EPN,CH,DSpr18-3269,6986,C,56.616667,-121.466667,1.0,...,0.000010,0.000010,0.000010,K4.West,0.995142,K5.West,0.994897,K6.West,0.989692,0.939222


# Removing samples with call rate < 85%

In [5]:
df_85 = df[df['call_rate']>0.85].reset_index(drop=True)
print(df_85.shape)

(1501, 136)


# Removing potentially mixed samples

In [24]:
strataAB = df_85[df_85['STRATA'].isin(['A','B'])].reset_index(drop=True)
strataAB['BestK4']
#strataAB['SPECIES_ID']

west_outliers = strataAB[(strataAB['SPECIES_ID'] != 'EPN') | (strataAB['BestK4'] != 'K4.West')]
west_outliers['sample']


strataI = df_85[df_85['STRATA'].isin(['I'])].reset_index(drop=True)
strataI['BestK4'].value_counts()
strataI['SPECIES_ID'].value_counts()

east_outliers = strataI[(strataI['SPECIES_ID'] != 'EPN') | (strataI['BestK4'] == 'K4.West')]
east_outliers

outliers = west_outliers['sample'].values.tolist() + east_outliers['sample'].values.tolist()
outliers

['E353-B3_5_9_7000_572',
 'E60-A_3_8_3_124',
 '6998_08',
 '4729_01',
 '6998_01',
 'E353-B3_3_12_6802_480']

# Filtered table

In [25]:
df_85_clean = df_85[~df_85['sample'].isin(outliers)].reset_index(drop=True)
print(df_85_clean.shape)

(1495, 136)


In [31]:
df_85_clean.to_csv("01_remove_mixed_admixture_metadata_clean.tsv", sep="\t", header=True, index=False)

In [32]:
df_samples_bad = df_85.loc[df_85['sample'].isin(outliers), :].reset_index(drop=True)
df_samples_bad

df_samples_bad.to_csv("01_remove_mixed_admixture_metadata_mixed.tsv", sep="\t", header=True, index=False)